[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/courses/dual_nature.ipynb)

# Dual nature of radiation and matter full course: the practice set, in Python

Twenty-eight exercises in three tiers: **tier 1** warm-ups (one idea, one step),
**tier 2** core problems (the level of the course's worked problems) and **tier 3** stretch problems (run a formula
backwards, combine two ideas, spot the trap). Each tier is harder than the last.

This notebook also has the course's **simulations and sums**: a photo-tube in which twenty thousand electrons are
flown across a gap and counted; the stopping potential read off that tube at five frequencies, with Planck's constant
from the slope of the line; a counter for the photons in a beam of light; the de Broglie wavelength of an electron and
a proton against the voltage that sped them up; electrons through two slits, landing one dot at a time; and the light
sensor's four metals on the visible spectrum. You can move all of them yourself: the light's **wavelength** and
**brightness**, the **voltage** between the plates, the **metal**, the beam's **power**, the **accelerating voltage**
and the **number of electrons** sent through the slits.

**How to use this notebook.** The grey boxes are cells. Click the first one and press **Shift + Enter** (or the ▶ Run
button) to run it. Run each cell in turn, from the top down. When sliders appear, move them and watch: the picture
redraws by itself. Change any number and run the cell again: nothing can break. It works in your browser (no install)
and in Google Colab.

**Typing answers.** Work the question out first (on paper, or in a cell of your own), then type your answer into the
box and press **check**: it tells you which parts are right. Press **show working** to see the steps. Give the answer
in the unit the question asks for, to two decimals or more. When a question asks for an answer "in units of 10⁻¹⁹ J",
type only the number in front: `3.66`. Big and small numbers in E notation: `3e8` for 3 × 10⁸, `6.63e-34` for
6.63 × 10⁻³⁴ (`3*10^8` works too). You can type the arithmetic: `1240/400`, `6.63e-34/(9.11e-31*2e6)`,
`1.23/sqrt(150)`. Numbers without thousands separators: `6200`, not `6,200`. For several answers, separate them with
commas, in the order the question lists them (the box's label shows the order). You can also put your answer between
the quotes in a cell's last line, `practice("P1.1", "")`, and run the cell: it checks straight away.

**Which constant to use.** The course's constants are rounded, so hc = 1240 eV nm, h = 4.14 × 10⁻¹⁵ eV s and
h = 6.63 × 10⁻³⁴ J s give answers that differ in the third figure. Every question names the one it wants.

**The cheat sheet.** Every formula and constant from the course on two pages:
[dual nature cheat sheet (PDF)](https://nammaschool.github.io/interactive-learning/cheatsheets/dual_nature_cheat_sheet.pdf).

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look, the constants, the formulas and the checker: run this cell first.
import ast, math, operator, re, textwrap
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from ipywidgets import (interact, FloatSlider, FloatLogSlider, IntSlider, Dropdown, Checkbox, Text, Button, Output, HBox,
                        VBox, Layout)

SLATE = "#1f2b28"                                     # the slate ground
CHALK, DIM, GHOST = "#ece6d6", "#93a097", "#3b4c47"   # electrons and curves in chalk, labels dim, ghost lines
SKY, MINT, CORAL, VIOLET, GOLD = "#6fb6bb", "#a0c27c", "#e3896b", "#b49ad6", "#e8c56b"
# photons VIOLET · electrons CHALK · current SKY · the wave picture's field CORAL · the one mark GOLD

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.axisbelow": True, "grid.color": GHOST, "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12,
    "lines.linewidth": 2.4, "figure.dpi": 100,
})

def footnote(fig, text="simulated · checked against the formula"):
    fig.text(0.99, 0.01, text, color=DIM, fontsize=10, ha="right", va="bottom", family="monospace")

# ---- the constants (the course's rounded values; every question names the one it wants) ----
H = 6.63e-34               # J s, Planck's constant
H_EV = 4.14e-15            # eV s, the same constant in electron volt seconds
C = 3.00e8                 # m/s, the speed of light
E = 1.60e-19               # C, the size of the electron's charge; also the joules in one electron volt
ME = 9.11e-31              # kg, the electron's mass
MP = 1.67e-27              # kg, the proton's mass
HC = 1240.0                # eV nm, the shortcut: h times c
EV = E                     # J in one electron volt

def close(a, b, rel=1e-4):
    """True when a simulated or summed value matches the formula (to a ten-thousandth by default)."""
    return abs(a - b) <= rel * max(abs(a), abs(b))

# ---- the course's formulas (energies in eV and light's wavelengths in nm, unless a line says otherwise) ----
def frequency(lam_nm): return C / (lam_nm * 1e-9)                                # ν = c / λ
def wavelength_nm(nu): return C / nu * 1e9                                       # λ = c / ν
def photon_ev(lam_nm): return HC / lam_nm                                        # E = hc / λ, in eV (λ in nm)
def photon_ev_f(nu): return H_EV * nu                                            # E = hν, in eV
def k_max(photon, phi): return max(0.0, photon - phi)                            # K max = hν − φ₀ (eV); none below threshold
def stopping(photon, phi): return k_max(photon, phi)                             # e V₀ = K max: volts, numerically the eV
def threshold_nm(phi): return HC / phi                                           # λ₀ = hc / φ₀
def threshold_hz(phi): return phi / H_EV                                         # ν₀ = φ₀ / h
def v0_line(nu, phi): return H_EV * nu - phi                                     # V₀ = (h/e) ν − φ₀/e, in volts
def photon_rate(P, lam_nm): return P / (photon_ev(lam_nm) * EV)                  # photons each second = power / (hν)
def momentum(lam_m): return H / lam_m                                            # p = h / λ (λ in metres)
def beam_force(P): return P / C                                                  # a beam fully absorbed: P / c
def speed_of(K_ev, m=ME): return math.sqrt(2 * K_ev * EV / m)                    # from K = ½ m v²
def kinetic_ev(v, m=ME): return 0.5 * m * v * v / EV                             # K = ½ m v², in eV
def de_broglie(m, v): return H / (m * v)                                         # λ = h / p = h / (m v), in metres
def de_broglie_v(V, m=ME, q=E): return H / math.sqrt(2 * m * q * V)              # λ = h / √(2 m e V), in metres
def lam_volts(V): return 1.23 / math.sqrt(V)                                     # the short form for an electron, in nm
def volts_for(lam_nm): return (1.23 / lam_nm) ** 2                               # the short form, turned round

# ---- the metals of the course's opening question, and the visible range ----
METALS = {"caesium": 1.95, "potassium": 2.29, "sodium": 2.75, "zinc": 3.63}     # work functions, in eV
VISIBLE = (400.0, 700.0)                                                         # nm, violet to red

# ---- reading a typed answer ----
OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv,
       ast.Pow: operator.pow, ast.USub: operator.neg, ast.UAdd: operator.pos}
FUNCS = {"sqrt": math.sqrt, "exp": math.exp, "cos": math.cos, "sin": math.sin}

def calc(text):
    """Plain arithmetic only: numbers (3e8 too), + − × ÷, powers, brackets, sqrt, exp, cos, sin (radians) and pi."""
    def ev(n):
        if isinstance(n, ast.Expression): return ev(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
        if isinstance(n, ast.Name) and n.id in ("pi", "π"): return math.pi
        if isinstance(n, ast.Name) and n.id == "e": return math.e
        if isinstance(n, ast.BinOp) and type(n.op) in OPS: return OPS[type(n.op)](ev(n.left), ev(n.right))
        if isinstance(n, ast.UnaryOp) and type(n.op) in OPS: return OPS[type(n.op)](ev(n.operand))
        if isinstance(n, ast.Call) and getattr(n.func, "id", "") in FUNCS and len(n.args) == 1:
            return FUNCS[n.func.id](ev(n.args[0]))
        raise ValueError(text)
    return float(ev(ast.parse(text, mode="eval")))

UNIT = (r"\s*(keV|MeV|eV|kg m/s|m/s|µJ|uJ|J|kV|V|nm|pm|µm|um|mm|cm|m|GHz|MHz|kHz|Hz|µA|uA|mA|A|nN|µN|uN|N|"
        r"mW|W|%|times)\s*$")

def read_part(s):
    s = s.strip().replace("−", "-").replace("–", "-").replace("×", "*").replace("÷", "/").replace("^", "**")
    s = s.replace("√", "sqrt")
    t = re.sub(UNIT, "", s).strip()
    try: return calc(t)
    except Exception: return s

def right(got, want, tol):
    return not isinstance(got, str) and abs(got - want) <= max(tol, 1e-7)

def show(v):
    return f"{v:,.10g}".replace(",", " ").replace("-", "−")

# ---- the exercises: the practice set's own (the course's questions and answers) ----
# @EX
EX = [
    {"id": 'P1.1', "tier": 1, "group": 'Photoelectric effect',
     "q": "Potassium's work function is 2.29 eV. What is that in joules, in units of 10⁻¹⁹ J? (1 eV = 1.60 × 10⁻¹⁹ J)",
     "parts": ['energy (× 10⁻¹⁹ J)'],
     "answer": [3.6639999999999997], "tol": [0.005], "function": '—',
     "hint": 'one electron volt is 1.60 × 10⁻¹⁹ joules: multiply',
     "working": '2.29 eV × 1.60 × 10⁻¹⁹ J/eV = 3.66 × 10⁻¹⁹ J'},
    {"id": 'P1.2', "tier": 1, "group": 'Photoelectric effect',
     "q": "In a photo-tube the current stops when the collector is 0.75 V below the emitter. What is the fastest electrons' kinetic energy, in eV?",
     "parts": ['K max (eV)'],
     "answer": [0.75], "tol": [0.005], "function": '—',
     "hint": 'K max = e V₀: an electron stopped by V₀ volts left with V₀ electron volts',
     "working": 'K max = e V₀ = 0.75 eV: one electron volt for each volt'},
    {"id": 'P1.3', "tier": 1, "group": 'Photoelectric effect',
     "q": "A photo-tube's saturation current is 4.8 µA. How many electrons cross each second, in units of 10¹³? (e = 1.60 × 10⁻¹⁹ C)",
     "parts": ['electrons a second (× 10¹³)'],
     "answer": [3.0], "tol": [0.005], "function": '/',
     "hint": "a current is charge each second: divide by one electron's charge",
     "working": '4.8 × 10⁻⁶ C/s ÷ 1.60 × 10⁻¹⁹ C = 3.0 × 10¹³ electrons a second'},
    {"id": 'P1.4', "tier": 1, "group": "Einstein's equation and photons",
     "q": 'What is the energy of one photon of orange light of wavelength 620 nm, in eV? (Use hc = 1240 eV nm.)',
     "parts": ['photon energy (eV)'],
     "answer": [2.0], "tol": [0.005], "function": '/',
     "hint": 'E = hc ÷ λ',
     "working": 'E = hc/λ = 1240 eV nm ÷ 620 nm = 2.00 eV'},
    {"id": 'P1.5', "tier": 1, "group": "Einstein's equation and photons",
     "q": 'What is the energy of one photon of light of frequency 5.0 × 10¹⁴ Hz, in eV? (Use h = 4.14 × 10⁻¹⁵ eV s.)',
     "parts": ['photon energy (eV)'],
     "answer": [2.0700000000000003], "tol": [0.005], "function": '*',
     "hint": 'E = h × ν',
     "working": 'E = hν = 4.14 × 10⁻¹⁵ eV s × 5.0 × 10¹⁴ Hz = 2.07 eV'},
    {"id": 'P1.6', "tier": 1, "group": "Einstein's equation and photons",
     "q": 'Photons of 3.10 eV fall on potassium (work function 2.29 eV). What is the greatest kinetic energy of the electrons that leave, in eV?',
     "parts": ['K max (eV)'],
     "answer": [0.81], "tol": [0.005], "function": '—',
     "hint": 'hν = φ₀ + K max',
     "working": 'K max = hν − φ₀ = 3.10 − 2.29 = 0.81 eV'},
    {"id": 'P1.7', "tier": 1, "group": "Einstein's equation and photons",
     "q": "Sodium's work function is 2.75 eV. What is its threshold wavelength, in nm? (Use hc = 1240 eV nm.)",
     "parts": ['threshold wavelength (nm)'],
     "answer": [450.90909090909093], "tol": [0.005], "function": '/',
     "hint": "at the threshold the photon's energy just equals the work function: hc/λ₀ = φ₀",
     "working": 'λ₀ = hc/φ₀ = 1240 ÷ 2.75 = 450.91 nm (blue light)'},
    {"id": 'P1.8', "tier": 1, "group": "Einstein's equation and photons",
     "q": 'What is the momentum of one photon of wavelength 500 nm, in units of 10⁻²⁷ kg m/s? (h = 6.63 × 10⁻³⁴ J s)',
     "parts": ['momentum (× 10⁻²⁷ kg m/s)'],
     "answer": [1.326], "tol": [0.005], "function": '/',
     "hint": 'p = h ÷ λ, with λ in metres',
     "working": 'p = h/λ = 6.63 × 10⁻³⁴ ÷ 500 × 10⁻⁹ = 1.33 × 10⁻²⁷ kg m/s'},
    {"id": 'P1.9', "tier": 1, "group": 'Matter waves',
     "q": "What is the de Broglie wavelength of an electron moving at 1.0 × 10⁶ m/s, in nm? (h = 6.63 × 10⁻³⁴ J s; the electron's mass is 9.11 × 10⁻³¹ kg)",
     "parts": ['wavelength (nm)'],
     "answer": [0.7277716794731065], "tol": [0.005], "function": '/',
     "hint": 'λ = h ÷ (m v)',
     "working": 'λ = h/(mv) = 6.63 × 10⁻³⁴ ÷ (9.11 × 10⁻³¹ × 1.0 × 10⁶) = 0.7278 nm'},
    {"id": 'P1.10', "tier": 1, "group": 'Matter waves',
     "q": 'An electron starts from rest and is accelerated through 150 V. What is its de Broglie wavelength, in nm? (Use λ = 1.23/√V nm.)',
     "parts": ['wavelength (nm)'],
     "answer": [0.1004290794541103], "tol": [0.005], "function": 'sqrt',
     "hint": 'λ = 1.23 ÷ √V, in nanometres',
     "working": 'λ = 1.23/√150 = 1.23 ÷ 12.25 = 0.1004 nm'},
    {"id": 'P2.1', "tier": 2, "group": 'Photoelectric effect',
     "q": "The fastest electrons leave a plate at 4.0 × 10⁵ m/s. What is their kinetic energy, in eV, and what stopping potential, in volts, would halt them? (the electron's mass is 9.11 × 10⁻³¹ kg; 1 eV = 1.60 × 10⁻¹⁹ J)",
     "parts": ['K max (eV)', 'stopping potential (V)'],
     "answer": [0.4555, 0.4555], "tol": [0.005, 0.005], "function": '—',
     "hint": 'K = ½ m v², then change joules to electron volts; e V₀ = K max',
     "working": 'K = ½mv² = ½ × 9.11 × 10⁻³¹ × (4.0 × 10⁵)² = 7.29 × 10⁻²⁰ J = 0.456 eV; e V₀ = K max, so V₀ = 0.456 V'},
    {"id": 'P2.2', "tier": 2, "group": "Einstein's equation and photons",
     "q": "Ultraviolet light of 350 nm falls on sodium (work function 2.75 eV). What is each photon's energy, in eV, and the greatest kinetic energy of the electrons, in eV? (Use hc = 1240 eV nm.)",
     "parts": ['photon energy (eV)', 'K max (eV)'],
     "answer": [3.5428571428571427, 0.7928571428571427], "tol": [0.005, 0.005], "function": '/',
     "hint": 'E = hc/λ, then hν = φ₀ + K max',
     "working": 'hν = 1240 ÷ 350 = 3.543 eV; K max = 3.543 − 2.75 = 0.793 eV'},
    {"id": 'P2.3', "tier": 2, "group": "Einstein's equation and photons",
     "q": 'Ultraviolet light of 250 nm falls on zinc (work function 3.63 eV). What is the stopping potential, in volts? (Use hc = 1240 eV nm.)',
     "parts": ['stopping potential (V)'],
     "answer": [1.33], "tol": [0.005], "function": '/',
     "hint": 'find K max first; an electron with K max electron volts is stopped by that many volts',
     "working": 'hν = 1240 ÷ 250 = 4.96 eV; K max = 4.96 − 3.63 = 1.33 eV, so V₀ = 1.33 V'},
    {"id": 'P2.4', "tier": 2, "group": "Einstein's equation and photons",
     "q": "Caesium's work function is 1.95 eV. What is its threshold frequency, in units of 10¹⁴ Hz? (Use h = 4.14 × 10⁻¹⁵ eV s.)",
     "parts": ['threshold frequency (× 10¹⁴ Hz)'],
     "answer": [4.7101449275362315], "tol": [0.005], "function": '/',
     "hint": 'at the threshold, hν₀ = φ₀',
     "working": 'ν₀ = φ₀/h = 1.95 ÷ 4.14 × 10⁻¹⁵ = 4.71 × 10¹⁴ Hz'},
    {"id": 'P2.5', "tier": 2, "group": "Einstein's equation and photons",
     "q": 'A lamp gives out 3.0 W of light of wavelength 600 nm. How many photons leave it each second, in units of 10¹⁸? (Use hc = 1240 eV nm.) (1 eV = 1.60 × 10⁻¹⁹ J)',
     "parts": ['photons a second (× 10¹⁸)'],
     "answer": [9.07258064516129], "tol": [0.005], "function": '/',
     "hint": 'photons each second = power ÷ the energy of one photon, in joules',
     "working": 'one photon: 1240 ÷ 600 = 2.067 eV = 3.307 × 10⁻¹⁹ J; 3.0 J/s ÷ 3.307 × 10⁻¹⁹ J = 9.07 × 10¹⁸ a second'},
    {"id": 'P2.6', "tier": 2, "group": "Einstein's equation and photons",
     "q": 'An FM radio station broadcasts at 100 MHz. What is the energy of one of its photons, in units of 10⁻⁷ eV? (Use h = 4.14 × 10⁻¹⁵ eV s.)',
     "parts": ['photon energy (× 10⁻⁷ eV)'],
     "answer": [4.140000000000001], "tol": [0.005], "function": '*',
     "hint": 'E = hν, with the frequency in hertz (100 MHz = 1e8 Hz)',
     "working": 'E = hν = 4.14 × 10⁻¹⁵ × 1.00 × 10⁸ = 4.14 × 10⁻⁷ eV'},
    {"id": 'P2.7', "tier": 2, "group": "Einstein's equation and photons",
     "q": "In a photo-tube experiment the stopping potential rises by 1.24 V when the light's frequency is raised by 3.0 × 10¹⁴ Hz. What is the slope h/e, in units of 10⁻¹⁵ V s, and the value of h it gives, in units of 10⁻³⁴ J s? (e = 1.60 × 10⁻¹⁹ C)",
     "parts": ['h/e (× 10⁻¹⁵ V s)', 'h (× 10⁻³⁴ J s)'],
     "answer": [4.133333333333333, 6.613333333333332], "tol": [0.005, 0.005], "function": '/',
     "hint": 'V₀ = (h/e)ν − φ₀/e: the slope of the line is h/e',
     "working": 'slope = 1.24 V ÷ 3.0 × 10¹⁴ Hz = 4.133 × 10⁻¹⁵ V s; h = slope × e = 6.613 × 10⁻³⁴ J s'},
    {"id": 'P2.8', "tier": 2, "group": 'Matter waves',
     "q": "What is the de Broglie wavelength of a proton moving at 1.0 × 10⁵ m/s, in picometres? (h = 6.63 × 10⁻³⁴ J s; the proton's mass is 1.67 × 10⁻²⁷ kg; 1 pm = 10⁻¹² m)",
     "parts": ['wavelength (pm)'],
     "answer": [3.9700598802395204], "tol": [0.005], "function": '/',
     "hint": 'λ = h ÷ (m v)',
     "working": 'λ = h/(mv) = 6.63 × 10⁻³⁴ ÷ (1.67 × 10⁻²⁷ × 1.0 × 10⁵) = 3.97 pm'},
    {"id": 'P2.9', "tier": 2, "group": 'Matter waves',
     "q": 'An electron is accelerated from rest through 400 V. What is its de Broglie wavelength, in picometres? (Use λ = 1.23/√V nm; 1 nm = 1000 pm.)',
     "parts": ['wavelength (pm)'],
     "answer": [61.5], "tol": [0.005], "function": 'sqrt',
     "hint": 'λ = 1.23 ÷ √V nanometres, then change the unit',
     "working": 'λ = 1.23/√400 = 1.23 ÷ 20 = 0.0615 nm = 61.5 pm'},
    {"id": 'P2.10', "tier": 2, "group": 'The light sensor',
     "q": "Potassium's work function is 2.29 eV. What is the longest wavelength of light that frees electrons from it, in nm? (Use hc = 1240 eV nm.)",
     "parts": ['threshold wavelength (nm)'],
     "answer": [541.4847161572052], "tol": [0.005], "function": '/',
     "hint": 'the longest wavelength has the smallest photon that still pays the work function',
     "working": 'λ₀ = hc/φ₀ = 1240 ÷ 2.29 = 541.48 nm: green light, and everything bluer, frees electrons; yellow and red do not'},
    {"id": 'P3.1', "tier": 3, "group": 'Photoelectric effect',
     "q": "The fastest photoelectrons from a plate travel at 8.0 × 10⁵ m/s. What stopping potential, in volts, halts them? (the electron's mass is 9.11 × 10⁻³¹ kg; e = 1.60 × 10⁻¹⁹ C)",
     "parts": ['stopping potential (V)'],
     "answer": [1.822], "tol": [0.005], "function": '—',
     "hint": 'e V₀ = ½ m v²: find the energy in joules, then divide by e',
     "working": 'K max = ½mv² = ½ × 9.11 × 10⁻³¹ × (8.0 × 10⁵)² = 2.915 × 10⁻¹⁹ J = 1.822 eV, so V₀ = 1.822 V'},
    {"id": 'P3.2', "tier": 3, "group": "Einstein's equation and photons",
     "q": "Light of 300 nm gives a stopping potential of 1.38 V on an unknown metal. What is the metal's work function, in eV? (Use hc = 1240 eV nm.)",
     "parts": ['work function (eV)'],
     "answer": [2.753333333333334], "tol": [0.005], "function": '/',
     "hint": 'hν = φ₀ + e V₀, run backwards',
     "working": 'hν = 1240 ÷ 300 = 4.133 eV; φ₀ = hν − e V₀ = 4.133 − 1.38 = 2.753 eV (sodium)'},
    {"id": 'P3.3', "tier": 3, "group": "Einstein's equation and photons",
     "q": 'Light of 400 nm gives a stopping potential of 0.81 V on a metal plate. What stopping potential, in volts, will light of 300 nm give on the same plate? (Use hc = 1240 eV nm.)',
     "parts": ['stopping potential (V)'],
     "answer": [1.8433333333333337], "tol": [0.005], "function": '/',
     "hint": 'use the first light to find the work function, then the second to find K max',
     "working": 'φ₀ = 1240/400 − 0.81 = 2.29 eV; then V₀ = 1240/300 − 2.29 = 1.843 V'},
    {"id": 'P3.4', "tier": 3, "group": "Einstein's equation and photons",
     "q": 'A 100 W lamp shines red light of 700 nm on sodium (work function 2.75 eV). What is the greatest kinetic energy of the electrons that leave, in eV? (Use hc = 1240 eV nm.)',
     "parts": ['K max (eV)'],
     "answer": [0.0], "tol": [0.005], "function": '/',
     "hint": "compare one photon's energy with the work function before anything else",
     "working": 'hν = 1240 ÷ 700 = 1.77 eV, less than 2.75 eV: no photon can pay the work function, so no electron leaves at all. The answer is 0, however powerful the lamp'},
    {"id": 'P3.5', "tier": 3, "group": "Einstein's equation and photons",
     "q": 'A 2.0 W beam of light is completely absorbed by a black card. What force does it put on the card, in nanonewtons? (c = 3.00 × 10⁸ m/s)',
     "parts": ['force (nN)'],
     "answer": [6.666666666666667], "tol": [0.005], "function": '/',
     "hint": "a photon's momentum is its energy over c; the force is the momentum arriving each second",
     "working": 'each photon brings momentum E/c, so each second the beam brings P/c = 2.0 ÷ 3.00 × 10⁸ = 6.67 nN'},
    {"id": 'P3.6', "tier": 3, "group": 'Matter waves',
     "q": 'Through what voltage must an electron be accelerated to have a de Broglie wavelength of 0.050 nm? (Use λ = 1.23/√V nm.)',
     "parts": ['voltage (V)'],
     "answer": [605.1599999999999], "tol": [0.005], "function": '—',
     "hint": 'turn λ = 1.23/√V round: √V = 1.23 ÷ λ',
     "working": '√V = 1.23 ÷ 0.050 = 24.6, so V = 24.6² = 605.16 V'},
    {"id": 'P3.7', "tier": 3, "group": 'Matter waves',
     "q": "A photon and an electron each have a wavelength of 0.20 nm. What is the photon's energy, in eV, and the electron's kinetic energy, in eV? (Use hc = 1240 eV nm.) (For the electron use λ = 1.23/√V nm.)",
     "parts": ['photon energy (eV)', "electron's kinetic energy (eV)"],
     "answer": [6200.0, 37.82249999999999], "tol": [0.005, 0.005], "function": '/',
     "hint": 'an electron accelerated through V volts has V electron volts',
     "working": 'photon: 1240 ÷ 0.20 = 6200 eV; electron: √V = 1.23 ÷ 0.20 = 6.15, V = 37.82 volts, so K = 37.82 eV: the same wavelength, very different energies'},
    {"id": 'P3.8', "tier": 3, "group": 'The light sensor',
     "q": 'Blue light of 450 nm and power 2.0 mW falls on a caesium plate (work function 1.95 eV). One photon in 500 frees an electron. What is K max, in eV, and the saturation current, in µA? (Use hc = 1240 eV nm.)',
     "parts": ['K max (eV)', 'saturation current (µA)'],
     "answer": [0.8055555555555556, 1.4516129032258065], "tol": [0.005, 0.005], "function": '/',
     "hint": "count the photons each second, take one in 500, and multiply by one electron's charge",
     "working": 'hν = 1240 ÷ 450 = 2.756 eV, K max = 0.806 eV; photons a second = 2.0 × 10⁻³ J/s ÷ (2.756 × 1.60 × 10⁻¹⁹ J) = 4.536e+15; one in 500 gives 9.073e+12 electrons a second, a current of 1.45 µA'},
]
# @END EX
Q = {e["id"]: e for e in EX}
TIER_NAMES = {1: "Warm-up", 2: "Core", 3: "Stretch"}
assert len(EX) == 28 and [sum(e["tier"] == t for e in EX) for t in TIER_NAMES] == [10, 10, 8]

def practice(k, answer=""):
    """Exercise k ("P1.4"): an answer box, a check button and a show-working button."""
    q = Q[k]
    n = len(q["answer"])
    box = Text(value=answer, placeholder="your answer" if n == 1 else f"{n} answers, separated by commas",
               layout=Layout(width="520px"))
    check, work, out = Button(description="check"), Button(description="show working"), Output()
    def on_check(_=None):
        out.clear_output()
        with out:
            got = [read_part(p) for p in re.split(r"[,;]", box.value) if p.strip()]
            if len(got) != n:
                print(f"✗ this one asks for {n} answer(s), in this order: {' · '.join(q['parts'])}. You typed {len(got)}."); return
            ok = [right(g_, w, t) for g_, w, t in zip(got, q["answer"], q["tol"])]
            if all(ok): print("✓ right" if n == 1 else "✓ all right"); return
            for good, part in zip(ok, q["parts"]):
                print(f"  {'✓' if good else '✗'} {part}" + ("" if good else ": not quite"))
            print(textwrap.fill(f"Hint: {q['hint']}", 110))
    def on_work(_=None):
        out.clear_output()
        with out:
            print(textwrap.fill(q["working"], 110))
            print("Answer: " + " · ".join(f"{p}: {show(a)}" for p, a in zip(q["parts"], q["answer"])))
    check.on_click(on_check); work.on_click(on_work)
    print(textwrap.fill(f"{k}. {q['q']}", 110))
    if n > 1: print(textwrap.fill("Answers, in order: " + " · ".join(q["parts"]), 110))
    display(VBox([HBox([box, check, work]), out]))
    if answer: on_check()

def answers(k):
    """The answers the practice set gives for exercise k (for the Python cells below to check against)."""
    return Q[k]["answer"]

def agrees(k, got):
    """True when every value in got is within exercise k's tolerance of its answer."""
    return len(got) == len(answers(k)) and all(right(g_, w, t) for g_, w, t in zip(got, answers(k), Q[k]["tol"]))

# the three routes to a photon's energy agree to the third figure, and no closer
assert abs(H * C / E * 1e9 / HC - 1) < 0.003 and abs(H / E / H_EV - 1) < 0.002
print("Ready: h =", H, "J s =", H_EV, "eV s; hc =", HC, "eV nm; c =", C, "m/s")

## Tier 1 · Warm-up

One idea, one step: electron volts into joules, the fastest electrons' energy from a stopping potential, electrons
counted from a current, a photon's energy from its wavelength or its frequency, Einstein's equation, a threshold
wavelength, a photon's momentum, and two de Broglie wavelengths.

In [ ]:
practice("P1.1")

In [ ]:
practice("P1.2")

In [ ]:
practice("P1.3")

In [ ]:
practice("P1.4")

In [ ]:
practice("P1.5")

In [ ]:
practice("P1.6")

In [ ]:
practice("P1.7")

In [ ]:
practice("P1.8")

In [ ]:
practice("P1.9")

In [ ]:
practice("P1.10")

### How you'd do this in Python

Each formula from the course is one short function in the first cell: `photon_ev(lam_nm)`, `photon_ev_f(nu)`,
`k_max(photon, phi)`, `threshold_nm(phi)`, `momentum(lam_m)`, `de_broglie(m, v)`, `lam_volts(V)` and the rest.
Energies are in electron volts and the wavelengths of light in nanometres. Two functions work in metres, `momentum`
and `de_broglie`, so 500 nm goes in as `500e-9` and the answer is changed to nanometres at the end. Each line below
computes one warm-up answer and checks it against the practice set.

In [ ]:
print(f"P1.4  E = {photon_ev(620):.2f} eV;  P1.7  λ₀ = {threshold_nm(2.75):.2f} nm;  "
      f"P1.9  λ = {de_broglie(ME, 1.0e6) * 1e9:.4f} nm")
assert agrees("P1.1", [2.29 * EV * 1e19]) and agrees("P1.2", [0.75])                    # one electron volt for each volt
assert agrees("P1.3", [4.8e-6 / E / 1e13]) and agrees("P1.4", [photon_ev(620)])
assert agrees("P1.5", [photon_ev_f(5.0e14)]) and agrees("P1.6", [k_max(3.10, 2.29)])
assert agrees("P1.7", [threshold_nm(2.75)]) and agrees("P1.8", [momentum(500e-9) * 1e27])
assert agrees("P1.9", [de_broglie(ME, 1.0e6) * 1e9]) and agrees("P1.10", [lam_volts(150)])
print("Every tier 1 answer, worked out in Python, matches the practice set.")

## The photo-tube: electrons flown across a gap and counted

Two flat metal plates face each other in a vacuum. Light falls on one of them, the emitter. The other, the collector,
catches the electrons that come across. This cell follows twenty thousand electrons, one by one (the course's own run
flies a hundred thousand; twenty thousand keeps it quick in a browser). Each electron takes one photon's energy hν,
pays the work function φ₀, and loses a random share of what is left on its way out of the metal. So the luckiest leave
with K max = hν − φ₀ and the rest leave with less. Each one starts from its own spot, in its own direction, and is
flown across the gap with or against the voltage between the plates. It is counted if it lands on the collector, and
the current is that count.

The curve is the current against the collector's voltage. Far to the right every electron is collected: that level is
the **saturation current**. To the left the collector pushes the electrons back, and at one voltage the current
stops. That voltage is the **stopping potential**. Here it is read off the simulated curve, then compared with
Einstein's equation. The reading comes out a percent or two below Einstein's value. The stopping potential is set by
the single luckiest electron, one that loses nothing on its way out and leaves straight across the gap, and among
twenty thousand electrons the best one is not quite that lucky.

Things to try:

- Make the light **brighter**. The whole curve grows taller, but its foot does not move: the stopping potential is
  the same.
- Change the **colour** (the wavelength). Now the foot does move: bluer light, bigger photons, a bigger stopping
  potential.
- Go **redder than the threshold** (past 541 nm for potassium). Nothing at all, however bright the light.
- Change the **metal**: a new work function, so a new threshold.

In [ ]:
TUBE = {"n": 20_000, "seed": 21, "r_emit": 0.5, "r_coll": 1.5}      # lengths in units of the gap between the plates
_rng = np.random.default_rng(TUBE["seed"])
_U, _MU = _rng.random(TUBE["n"]), _rng.random(TUBE["n"])              # the share of energy lost; cos θ of the direction
_R0, _AZ0, _AZ = (TUBE["r_emit"] * np.sqrt(_rng.random(TUBE["n"])), 2 * math.pi * _rng.random(TUBE["n"]),
                  2 * math.pi * _rng.random(TUBE["n"]))
_X0, _Y0 = _R0 * np.cos(_AZ0), _R0 * np.sin(_AZ0)                     # each electron's starting spot on the emitter

def tube_fraction(kmax_ev, V):
    """The share of the emitted electrons that land on the collector at voltage V (each flown across the gap).
    V is the collector's voltage above the emitter's: positive pulls electrons across, negative opposes them."""
    if kmax_ev <= 0: return 0.0
    K = kmax_ev * (1 - _U)                               # K = K max (1 − u): deeper electrons lose more
    kz, kt = K * _MU ** 2, K * (1 - _MU ** 2)            # the energy of the motion across the gap, and sideways
    ok = kz + V > 0                                      # it reaches the far plate at all
    tau = 2.0 / (np.sqrt(kz) + np.sqrt(np.where(ok, kz + V, 1.0)))       # the flight time, in units of d / √(2e/m)
    s = np.sqrt(kt) * tau                                # how far it drifts sideways on the way
    x, y = _X0 + s * np.cos(_AZ), _Y0 + s * np.sin(_AZ)
    return float(np.mean(ok & (x * x + y * y <= TUBE["r_coll"] ** 2)))

def tube_current(photon, phi, intensity, V):
    """The current, in units of the saturation current at brightness 1: one electron for each of a fixed share of the
    photons, so the number leaving each second is in proportion to the brightness."""
    return intensity * tube_fraction(k_max(photon, phi), V)

def tube_stop(photon, phi, intensity=1.0, step=0.001):
    """The stopping potential READ OFF the simulated tube: the smallest opposing voltage (to 1 mV) at which no electron
    arrives. None when no electron leaves at all."""
    km = k_max(photon, phi)
    if km <= 0: return None
    v = 0.0
    while tube_current(photon, phi, intensity, -v) > 0: v += max(step, 0.02 * km)
    lo, hi = max(0.0, v - max(step, 0.02 * km)), v
    while hi - lo > step:
        mid = (lo + hi) / 2
        lo, hi = (mid, hi) if tube_current(photon, phi, intensity, -mid) > 0 else (lo, mid)
    return hi

def tube_picture(metal="potassium", wavelength_nm=400.0, brightness=1.0, volts=0.0):
    phi, photon = METALS[metal], photon_ev(wavelength_nm)
    km = k_max(photon, phi)
    v0 = tube_stop(photon, phi, brightness)              # None below the threshold
    vs = np.linspace(min(-2.5, -(v0 or 0.0) - 0.5), 4.0, 131)
    curve = [tube_current(photon, phi, brightness, v_) for v_ in vs]
    here, sat = tube_current(photon, phi, brightness, volts), tube_current(photon, phi, brightness, 200.0)
    fig, ax = plt.subplots(figsize=(10, 4.4))
    fig.subplots_adjust(left=0.09, right=0.98, top=0.84, bottom=0.16)
    ax.plot(vs, curve, color=SKY, label="current")
    if v0 is None:
        ax.text(0.5, 0.55, "no electrons: the photon is smaller than the work function", transform=ax.transAxes,
                ha="center", color=CHALK)
        title = f"{metal} · {wavelength_nm:g} nm · the photon has {photon:.2f} eV, the work function is {phi:.2f} eV"
    else:
        ax.axhline(sat, color=GHOST, linestyle="--", linewidth=1.4)
        ax.text(3.95, sat + 0.05, "saturation current", color=DIM, ha="right", va="bottom", fontsize=11)
        ax.axvline(-v0, color=CHALK, linestyle=":", linewidth=1.6, label=f"the current stops at −{v0:.2f} V")
        title = (f"{metal} · {wavelength_nm:g} nm · K max = {photon:.2f} − {phi:.2f} = {km:.2f} eV · "
                 f"stopping potential read off: {v0:.2f} V")
    ax.plot([volts], [here], "o", color=GOLD, ms=10, label=f"at {volts:+.2f} V the current is {here:.2f}".replace("-", "−"))
    ax.set_ylim(-0.1, 3.3); ax.set_xlabel("collector's voltage, in V")
    ax.set_ylabel("current (saturation at brightness 1 = 1)"); ax.legend(frameon=False, loc="upper left")
    fig.suptitle(title, x=0.03, ha="left", color=GOLD)
    footnote(fig); plt.show()
    if v0 is None:
        print(f"hν = {photon:.2f} eV is less than φ₀ = {phi:.2f} eV. No electrons: the photon is smaller than the work "
              f"function.")
    else:
        print(f"K max = hν − φ₀ = {photon:.2f} − {phi:.2f} = {km:.2f} eV, so Einstein's equation gives a stopping "
              f"potential of {km:.2f} V.")
        print(f"Read off the simulated curve: the current stops when the collector is {v0:.2f} V below the emitter.")
    return v0, sat, here

v0_k, sat_1, _ = tube_picture("potassium", 400.0, 1.0, 0.0)
violet = photon_ev(400.0)
assert close(v0_k, k_max(violet, 2.29), 0.03) and v0_k <= k_max(violet, 2.29)     # within 3 % of hν − φ₀, never above it
assert sat_1 == 1.0 and tube_current(violet, 2.29, 2.0, 200.0) == 2 * sat_1       # twice as bright: twice the current
assert tube_stop(violet, 2.29, 2.0) == tube_stop(violet, 2.29, 1.0) == v0_k       # and the same stopping potential
assert all(tube_current(photon_ev(650.0), 2.29, I_, v_) == 0.0 for I_ in (1.0, 1000.0, 1e6) for v_ in (-1.0, 0.0, 4.0, 200.0))
assert tube_stop(photon_ev(650.0), 2.29) is None                                  # red light on potassium: nothing
assert tube_picture("potassium", 650.0, 3.0, 4.0) == (None, 0.0, 0.0)             # however bright, however hard it is pulled
assert agrees("P1.6", [k_max(3.10, 2.29)])

In [ ]:
interact(tube_picture, metal=Dropdown(options=list(METALS), value="potassium", description="metal"),
         wavelength_nm=FloatSlider(value=400.0, min=250.0, max=700.0, step=5.0, description="light (nm)"),
         brightness=FloatSlider(value=1.0, min=0.5, max=3.0, step=0.5, description="brightness"),
         volts=FloatSlider(value=0.0, min=-2.5, max=4.0, step=0.05, description="voltage (V)"));

## Tier 2 · Core

The level of the course's worked problems: a stopping potential from a speed, Einstein's equation from a wavelength,
a threshold frequency, the photons in a lamp's light, a radio photon, h from a slope, the de Broglie wavelength of a
proton and of an electron sped up through a voltage, and potassium's threshold.

In [ ]:
practice("P2.1")

In [ ]:
practice("P2.2")

In [ ]:
practice("P2.3")

In [ ]:
practice("P2.4")

In [ ]:
practice("P2.5")

In [ ]:
practice("P2.6")

In [ ]:
practice("P2.7")

In [ ]:
practice("P2.8")

In [ ]:
practice("P2.9")

In [ ]:
practice("P2.10")

### How you'd do this in Python

Two-part answers are just two lines. An electron that leaves with K electron volts is stopped by K volts, so
`stopping(photon, phi)` and `k_max(photon, phi)` give the same number, one in volts and one in electron volts.

In [ ]:
K_21 = kinetic_ev(4.0e5)
slope_27 = 1.24 / 3.0e14                                    # volts for each hertz: the slope h/e
print(f"P2.1  K max = {K_21:.3f} eV, V₀ = {K_21:.3f} V;  P2.7  slope = {slope_27:.3e} V s, h = {slope_27 * E:.3e} J s")
assert agrees("P2.1", [K_21, K_21]) and agrees("P2.2", [photon_ev(350), k_max(photon_ev(350), 2.75)])
assert agrees("P2.3", [stopping(photon_ev(250), 3.63)]) and agrees("P2.4", [threshold_hz(1.95) / 1e14])
assert agrees("P2.5", [photon_rate(3.0, 600) / 1e18]) and agrees("P2.6", [photon_ev_f(100e6) * 1e7])
assert agrees("P2.7", [slope_27 * 1e15, slope_27 * E * 1e34]) and agrees("P2.8", [de_broglie(MP, 1.0e5) * 1e12])
assert agrees("P2.9", [lam_volts(400) * 1000]) and agrees("P2.10", [threshold_nm(2.29)])
print("Every tier 2 answer, worked out in Python, matches the practice set.")

## The stopping potential against frequency: h from a slope

Einstein's equation says e V₀ = hν − φ₀. So a graph of the stopping potential against the light's frequency is a
straight line, V₀ = (h/e) ν − φ₀/e. Its slope is h/e, whatever the metal. This cell reads the stopping potential off
the simulated tube at five frequencies, adds a small scatter to each (they are **illustrative readings**, not a real
experiment's), and fits a straight line through them. The slope times e gives Planck's constant. Where the line
meets the frequency axis is the **threshold frequency**, and where it would meet the voltage axis gives the work
function.

Choose another metal: the line moves sideways, to a new threshold frequency, but its slope stays the same. The
other metals' lines are drawn faintly, and they are parallel. (Every reading off this tube is a percent or two low, as
the last section said, so the slope comes out a percent or two under the course's h/e.)

In [ ]:
FITTED = {}

def line_fit(metal):
    """Five stopping potentials read off the simulated tube, each with a small seeded scatter (illustrative readings),
    and the straight line fitted through them."""
    if metal not in FITTED:
        phi = METALS[metal]
        first = math.ceil((threshold_hz(phi) + 0.5e14) / 0.5e14) * 0.5e14        # start a little above the threshold
        nu = np.array([first + k * 1.0e14 for k in range(5)])
        rng = np.random.default_rng(2116 + list(METALS).index(metal))
        v0 = np.array([tube_stop(photon_ev_f(f_), phi) + float(rng.normal(0.0, 0.008)) for f_ in nu])
        slope, icpt = (float(v_) for v_ in np.polyfit(nu, v0, 1))
        FITTED[metal] = {"nu": nu, "v0": v0, "slope": slope, "h": slope * E, "phi": -icpt, "nu0": -icpt / slope}
    return FITTED[metal]

def slope_picture(metal="potassium"):
    fit = line_fit(metal)
    fig, ax = plt.subplots(figsize=(10, 4.6))
    fig.subplots_adjust(left=0.09, right=0.98, top=0.84, bottom=0.16)
    for name in METALS:
        f_ = line_fit(name)
        xs = np.array([f_["nu0"], 14.5e14])
        ax.plot(xs / 1e14, f_["slope"] * (xs - f_["nu0"]), color=GOLD if name == metal else DIM,
                linewidth=2.4 if name == metal else 1.3)
        ax.text(14.65, f_["slope"] * (14.5e14 - f_["nu0"]), name, ha="left", va="center", fontsize=11,
                color=CHALK if name == metal else DIM)
    ax.plot(fit["nu"] / 1e14, fit["v0"], "o", color=CHALK, ms=8, label="stopping potentials read off the tube")
    ax.plot([fit["nu0"] / 1e14], [0.0], "|", color=CHALK, ms=16, mew=2, label="threshold frequency, from the line")
    ax.set_xlim(4.0, 16.2); ax.set_ylim(0.0, 4.6)
    ax.set_xlabel("frequency of the light, in 10¹⁴ Hz"); ax.set_ylabel("stopping potential, in V")
    ax.legend(frameon=False, loc="upper left")
    fig.suptitle(f"{metal}: slope {fit['slope']:.3e} V s · h = slope × e = {fit['h']:.3e} J s", x=0.03, ha="left", color=GOLD)
    footnote(fig, "illustrative readings off the simulated tube · line fitted by least squares"); plt.show()
    print(f"slope of the line, h/e: {fit['slope']:.3e} V s")
    print(f"h = slope × e = {fit['h']:.3e} J s (the course's value: {H:.2e} J s)")
    print(f"work function, from the line: {fit['phi']:.2f} eV ({metal}: {METALS[metal]:.2f} eV)")
    print(f"threshold frequency, from the line: {fit['nu0']:.2e} Hz (φ₀/h = {threshold_hz(METALS[metal]):.2e} Hz)")
    return fit["slope"], fit["h"]

slope_k, h_k = slope_picture("potassium")
fit_k = line_fit("potassium")
three = [line_fit(m_) for m_ in ("caesium", "potassium", "sodium")]
assert slope_k == fit_k["slope"] and h_k == slope_k * E                            # h is the slope times e
assert all(close(f_["h"], H, 0.03) for f_ in three)                                # h from each slope, within 3 %
assert max(f_["slope"] for f_ in three) / min(f_["slope"] for f_ in three) - 1 < 0.03       # one slope for every metal
assert close(fit_k["phi"], 2.29, 0.05) and close(fit_k["nu0"], threshold_hz(2.29), 0.03)
assert agrees("P2.4", [threshold_hz(1.95) / 1e14])

In [ ]:
interact(slope_picture, metal=Dropdown(options=list(METALS), value="potassium", description="metal"));

## Counting photons

A beam of light is a stream of photons, each carrying hν. So the number of photons each second is the beam's power
divided by one photon's energy. Each photon also carries momentum, h/λ. A black card that absorbs the beam takes all
of that momentum, and the momentum arriving each second is the force on the card. It comes to the beam's power
divided by c. The starting values are a 1 mW green laser pointer.

Slide the power: more photons each second, and each one the same size. Slide the wavelength: the photons change size,
so the same power needs a different number of them.

In [ ]:
def photon_counter(power_mW=1.0, wavelength_nm=532.0):
    P = power_mW * 1e-3
    photon = photon_ev(wavelength_nm)
    rate, p, force = photon_rate(P, wavelength_nm), momentum(wavelength_nm * 1e-9), beam_force(P)
    print(f"{power_mW:g} mW of light of {wavelength_nm:g} nm")
    print(f"  one photon's energy:            {photon:.2f} eV = {photon * EV:.2e} J")
    print(f"  photons each second:            {rate:.2e}")
    print(f"  one photon's momentum, h/λ:     {p:.2e} kg m/s")
    print(f"  force on a card that absorbs it: {force:.2e} N (photons each second × momentum: {rate * p:.2e} N)")
    return rate, p, force

rate_g, p_g, force_g = photon_counter(1.0, 532.0)
assert close(rate_g * p_g, force_g, 5e-3)                 # photons × momentum = P/c (the rounded constants: within 0.5 %)
assert round(photon_ev(532.0), 2) == 2.33 and round(rate_g / 1e15, 1) == 2.7 and round(force_g * 1e12, 1) == 3.3
assert photon_rate(2e-3, 532.0) == 2 * rate_g             # brighter means more photons, never bigger ones
assert agrees("P3.5", [beam_force(2.0) * 1e9]) and agrees("P2.5", [photon_rate(3.0, 600) / 1e18])

In [ ]:
interact(photon_counter, power_mW=FloatSlider(value=1.0, min=0.5, max=100.0, step=0.5, description="power (mW)"),
         wavelength_nm=FloatSlider(value=532.0, min=250.0, max=1000.0, step=1.0, description="light (nm)"));

## Tier 3 · Stretch

Run a formula backwards, combine two ideas, or spot the trap: a stopping potential from a speed, a work function from
a stopping potential, two colours on one plate, a powerful red lamp, the push of a beam, the voltage for a chosen
wavelength, a photon and an electron of the same wavelength, and the light sensor's current.

In [ ]:
practice("P3.1")

In [ ]:
practice("P3.2")

In [ ]:
practice("P3.3")

In [ ]:
practice("P3.4")

In [ ]:
practice("P3.5")

In [ ]:
practice("P3.6")

In [ ]:
practice("P3.7")

In [ ]:
practice("P3.8")

### How you'd do this in Python

Running a formula backwards is one line of algebra, then one line of Python. The trap in P3.4: compare one photon's
energy with the work function before anything else. `k_max` does that for you and gives 0 when the photon is too
small, however powerful the lamp.

In [ ]:
phi_33 = photon_ev(400) - 0.81                              # the first light gives the plate's work function
blue = photon_ev(450)
print(f"P3.3  φ₀ = {phi_33:.2f} eV, then V₀ = {photon_ev(300) - phi_33:.2f} V;  P3.4  K max = {k_max(photon_ev(700), 2.75):g} eV")
assert agrees("P3.1", [kinetic_ev(8.0e5)]) and agrees("P3.2", [photon_ev(300) - 1.38])
assert agrees("P3.3", [photon_ev(300) - phi_33]) and agrees("P3.4", [k_max(photon_ev(700), 2.75)])
assert agrees("P3.5", [beam_force(2.0) * 1e9]) and agrees("P3.6", [volts_for(0.050)])
assert agrees("P3.7", [photon_ev(0.20), volts_for(0.20)])
assert agrees("P3.8", [k_max(blue, 1.95), photon_rate(2.0e-3, 450) / 500 * E * 1e6])
print("Every tier 3 answer, worked out in Python, matches the practice set.")

## Matter waves: the de Broglie wavelength

De Broglie's idea: anything with momentum p has a wavelength, λ = h/p. An electron that starts from rest and crosses
V volts gains V electron volts, so its momentum is √(2 m e V) and its **de Broglie wavelength** is h/√(2 m e V). For
an electron that comes to 1.23/√V nanometres. The chart has powers of ten on both axes, so the rule "four times the
voltage, half the wavelength" is a straight line. A proton through the same voltage is about 43 times shorter, because
it is about 1,800 times heavier.

Slide the voltage and compare the electron's wavelength with green light's 500 nm. Above about 10,000 volts the
electron is moving so fast that this plain formula needs Einstein's correction: at 10,000 volts it is already half a
percent too long.

In [ ]:
def matter_picture(volts=100.0):
    lam = de_broglie_v(volts) * 1e9                          # the electron's wavelength, in nm
    Vs = np.logspace(0, 4, 120)
    fig, ax = plt.subplots(figsize=(10, 4.6))
    fig.subplots_adjust(left=0.1, right=0.98, top=0.84, bottom=0.16)
    ax.loglog(Vs, [de_broglie_v(v_) * 1e9 for v_ in Vs], color=CHALK, label="electron")
    ax.loglog(Vs, [de_broglie_v(v_, MP) * 1e9 for v_ in Vs], color=DIM, linestyle="--", label="proton")
    ax.axhline(0.1, color=GHOST, linewidth=1.4)
    ax.text(1.1, 0.108, "about the size of an atom, 0.1 nm", color=DIM, fontsize=11, va="bottom")
    ax.plot([volts], [lam], "o", color=GOLD, ms=10)
    ax.set_xlabel("accelerating voltage, in V"); ax.set_ylabel("de Broglie wavelength, in nm")
    ax.legend(frameon=False, loc="lower left")
    fig.suptitle(f"an electron through {volts:,.0f} V: λ = {lam:.4f} nm · {500.0 / lam:,.0f} times shorter than green light",
                 x=0.03, ha="left", color=GOLD)
    footnote(fig, "our own example · computed"); plt.show()
    print(f"electron through {volts:,.0f} V: λ = h/√(2meV) = {lam:.4f} nm (the short form 1.23/√V gives {lam_volts(volts):.4f} nm)")
    print(f"that is {500.0 / lam:,.0f} times shorter than green light of 500 nm")
    print(f"a proton through the same voltage: {de_broglie_v(volts, MP) * 1e9:.5f} nm")
    return lam

lam_100 = matter_picture(100.0)
ball = de_broglie(0.058, 50.0)                               # a served tennis ball: 58 g at 50 m/s
print(f"a served tennis ball (58 g at 50 m/s): λ = {ball:.1e} m, far smaller than any nucleus")
assert round(lam_100, 4) == 0.1228 and round(lam_100, 3) == 0.123 and round(ball * 1e34, 1) == 2.3
assert close(de_broglie_v(400.0), de_broglie_v(100.0) / 2, 1e-12)                 # four times the voltage: half the wavelength
assert close(lam_volts(100.0), lam_100, 2e-3) and round(math.sqrt(MP / ME)) == 43
too_long = math.sqrt(1 + 1.0e4 * EV / (2 * ME * C * C)) - 1                       # the plain formula's error at 10,000 V
assert 0.004 < too_long < 0.006
assert agrees("P1.10", [lam_volts(150)]) and agrees("P3.6", [volts_for(0.050)])

In [ ]:
interact(matter_picture, volts=FloatLogSlider(value=100.0, base=10, min=0.0, max=4.0, step=0.02, description="voltage (V)"));

## Electrons through two slits, one dot at a time

Electrons sped up through 600 volts have a de Broglie wavelength of 0.050 nm. They are sent, one at a time, at two
slits a micrometre apart, each 0.2 µm wide, with a screen a metre behind. Every electron lands as **one dot**, in one
place. Where it lands is drawn at random from the pattern that a wave of the electron's wavelength makes. The upper
panel is the screen. The lower panel counts the dots in strips 5 µm wide.

Start with ten electrons: the dots look scattered. Slide to a few hundred, then to six thousand: bright and dark
bands build up, dot by dot. The spacing of the bands is then measured from the dots alone, and it matches λL/d, the
spacing for a wave of that wavelength, about 50 µm.

Untick **wave** and the electrons are thrown like tiny bullets, with one pile behind each slit. The slits are only a
micrometre apart and each pile is hundreds of micrometres wide, so the two piles lie on top of each other as one
smooth hump, with no bands.

In [ ]:
SLIT = {"V": 600.0, "d": 1.0e-6, "a": 0.2e-6, "L": 1.0, "n": 6000, "seed": 1927}
SLIT["lam"] = de_broglie_v(SLIT["V"])                                            # 5.0e-11 m
SLIT["band"] = SLIT["lam"] * SLIT["L"] / SLIT["d"] * 1e6                         # µm between neighbouring bright bands: 50
SLIT["half"] = SLIT["lam"] * SLIT["L"] / SLIT["a"] * 1e6                         # µm to the edge of one slit's spread: 250

def slit_pattern(x_um, wave=True):
    """How often an electron lands at x (µm from the middle), not normalised: with wave=True the two-slit pattern of a
    wave of the electron's de Broglie wavelength; with wave=False two piles, one behind each slit's spread, simply added
    (what tiny bullets would do)."""
    x = np.asarray(x_um, dtype=float)
    env = np.sinc(x / SLIT["half"]) ** 2                                         # one slit's spread (np.sinc(t) = sin(πt)/(πt))
    return env * (2 * np.cos(math.pi * x / SLIT["band"]) ** 2 if wave else 1.0)

def slit_dots(n=None, wave=True, seed=None):
    """n landing places (x, y) in µm, each drawn at random from slit_pattern; y is uniform (the slits are long)."""
    n = n or SLIT["n"]
    rng = np.random.default_rng(SLIT["seed"] if seed is None else seed)
    grid = np.linspace(-SLIT["half"], SLIT["half"], 4001)
    cdf = np.cumsum(slit_pattern(grid, wave)); cdf /= cdf[-1]
    return np.interp(rng.random(n), cdf, grid), rng.uniform(-60.0, 60.0, n)

def slit_measure(x, lo=30.0, hi=80.0, steps=501):
    """The band spacing MEASURED from the dots alone: the spacing s whose ripple cos(2πx/s) the dots follow most
    strongly; returns (s, strength), strength from 0 (no bands) to about 0.5 (clean bands)."""
    s = np.linspace(lo, hi, steps)
    amp = np.abs(np.exp(2j * math.pi * x[None, :] / s[:, None]).mean(axis=1))
    k = int(np.argmax(amp))
    return float(s[k]), float(amp[k])

DOTS_X, DOTS_Y = slit_dots()                                 # six thousand electrons, in the order they land
PILES_X, PILES_Y = slit_dots(wave=False)                     # the same number thrown as tiny bullets

def slits_picture(dots=200, wave=True):
    n = int(dots)
    x, y = (DOTS_X[:n], DOTS_Y[:n]) if wave else (PILES_X[:n], PILES_Y[:n])
    spacing, strength = slit_measure(x)
    fig, (top, low) = plt.subplots(2, 1, figsize=(10, 5.6), sharex=True, gridspec_kw={"height_ratios": [1, 1.25]})
    fig.subplots_adjust(left=0.09, right=0.98, top=0.86, bottom=0.13, hspace=0.12)
    top.scatter(x, y, s=max(3.0, min(36.0, 4000.0 / n)), color=CHALK, linewidths=0)
    top.set_ylim(-66, 66); top.set_yticks([]); top.grid(False); top.set_ylabel("the screen")
    edges = np.linspace(-SLIT["half"], SLIT["half"], 101)                        # strips 5 µm wide
    low.hist(x, bins=edges, color=CHALK)
    grid = np.linspace(-SLIT["half"], SLIT["half"], 1001)
    shape = slit_pattern(grid, wave)
    shape = shape * n * (edges[1] - edges[0]) / (shape.sum() * (grid[1] - grid[0]))      # the dots expected in each strip
    low.plot(grid, shape, color=CORAL, linewidth=1.6,
             label="a wave of the de Broglie wavelength" if wave else "two piles, simply added")
    low.set_xlim(-SLIT["half"], SLIT["half"]); low.set_xlabel("position on the screen, in µm")
    low.set_ylabel("electrons in each 5 µm"); low.legend(frameon=False, loc="upper right")
    if not wave: title = f"{n:,} electrons as tiny bullets: no bands (ripple strength {strength:.2f})"
    elif n >= 100 and strength > 0.3:
        title = f"{n:,} electrons: bands {spacing:.1f} µm apart, measured from the dots (λL/d = {SLIT['band']:.1f} µm)"
    else: title = f"{n:,} electrons: too few dots to measure the bands yet"
    fig.suptitle(title, x=0.03, ha="left", color=GOLD)
    footnote(fig); plt.show()
    return spacing, strength

band_6000, strength_6000 = slits_picture(6000, True)
assert round(SLIT["lam"] * 1e9, 3) == 0.05 and round(SLIT["band"]) == 50 and round(SLIT["half"], -1) == 250
assert close(band_6000, SLIT["band"], 0.02) and strength_6000 > 0.3           # measured from the dots: within 2 % of λL/d
assert slits_picture(6000, False)[1] < 0.06                                   # two piles: no bands
assert float(np.mean(np.abs(((DOTS_X / SLIT["band"]) % 1.0) - 0.5) < 0.1)) < 0.03    # almost no dot lands near a dark line
assert agrees("P3.6", [volts_for(0.050)])                                     # 0.050 nm needs about 600 V

In [ ]:
interact(slits_picture, dots=IntSlider(value=200, min=10, max=6000, step=10, description="electrons"),
         wave=Checkbox(value=True, description="wave"));

## The light sensor: four metals on the visible spectrum

The course's opening question. A light sensor needs a metal plate that gives up electrons when visible light falls on
it. Four metals are on offer. A metal answers to light only when one photon can pay its work function, so each metal
has a longest wavelength it answers to, λ₀ = hc/φ₀. The bar is the visible spectrum, 400 to 700 nm, drawn in
approximate colour, with each metal's threshold wavelength marked on it. Zinc's is at 342 nm, off the bar in the
ultraviolet: zinc is blind to every colour we can see. Caesium answers to the most of the spectrum, so caesium is the
sensor's metal.

Slide the light's wavelength: the cell lists which metals give up electrons, and the fastest electrons' energy.

In [ ]:
def nm_rgb(lam):
    """A rough screen colour for a wavelength of visible light, in nm (a common approximation, not a measured one)."""
    if lam < 440: rgb = ((440 - lam) / 60, 0.0, 1.0)
    elif lam < 490: rgb = (0.0, (lam - 440) / 50, 1.0)
    elif lam < 510: rgb = (0.0, 1.0, (510 - lam) / 20)
    elif lam < 580: rgb = ((lam - 510) / 70, 1.0, 0.0)
    elif lam < 645: rgb = (1.0, (645 - lam) / 65, 0.0)
    else: rgb = (1.0, 0.0, 0.0)
    dim = 0.3 + 0.7 * (lam - 380) / 40 if lam < 420 else (0.3 + 0.7 * (750 - lam) / 50 if lam > 700 else 1.0)
    return tuple((max(0.0, c_) * dim) ** 0.8 for c_ in rgb)

def sees(metal, lam_nm): return photon_ev(lam_nm) > METALS[metal]

def sensor_picture(light_nm=450.0):
    photon = photon_ev(light_nm)
    fig, ax = plt.subplots(figsize=(10.5, 4.4))
    fig.subplots_adjust(left=0.03, right=0.97, top=0.84, bottom=0.17)
    lams = np.linspace(VISIBLE[0], VISIBLE[1], 301)
    ax.imshow([[nm_rgb(l_) for l_ in lams]], extent=(VISIBLE[0], VISIBLE[1], 4.0, 4.9), aspect="auto")
    ax.text(360, 4.45, "ultraviolet", color=DIM, ha="center", va="center", fontsize=11)
    out = {}
    for k, (metal, phi) in enumerate(METALS.items()):
        lam0, y, on = threshold_nm(phi), 3.1 - k, sees(metal, light_nm)
        ax.plot([320, lam0], [y, y], color=CHALK if on else DIM, linewidth=7, solid_capstyle="butt")
        ax.plot([lam0, lam0], [y, 4.9 if lam0 >= VISIBLE[0] else y + 0.3], color=CHALK, linewidth=1.2, linestyle=":")
        ax.text(lam0 + 5, y, f"{metal}: up to {lam0:.0f} nm", va="center", color=CHALK if on else DIM, fontsize=12)
        out[metal] = k_max(photon, phi) if on else None
    ax.axvline(light_nm, color=GOLD)
    ax.set_xlim(320, 720); ax.set_ylim(-0.6, 5.0); ax.set_yticks([]); ax.grid(False)
    ax.set_xlabel("wavelength of the light, in nm")
    yes = [m_ for m_ in METALS if out[m_] is not None]
    leave = ("all four metals" if len(yes) == 4 else ", ".join(yes) if yes else "no metal of the four")
    fig.suptitle(f"light of {light_nm:g} nm: each photon has {photon:.2f} eV · electrons leave {leave}", x=0.03,
                 ha="left", color=GOLD)
    footnote(fig, "drawn in approximate colour · thresholds from λ₀ = hc/φ₀"); plt.show()
    for metal, phi in METALS.items():
        print(f"  {metal:<10} work function {phi:.2f} eV, threshold {threshold_nm(phi):.0f} nm: " +
              (f"electrons leave, K max = {out[metal]:.2f} eV" if out[metal] is not None else "no electrons"))
    return yes

seen_450 = sensor_picture(450.0)
assert [round(threshold_nm(v_)) for v_ in METALS.values()] == [636, 541, 451, 342]
assert seen_450 == ["caesium", "potassium", "sodium"]                         # blue light: three of the four answer
assert not any(sees("zinc", l_) for l_ in range(400, 701))                    # zinc answers to no visible wavelength
share = {m_: max(0.0, min(VISIBLE[1], threshold_nm(v_)) - VISIBLE[0]) / (VISIBLE[1] - VISIBLE[0]) for m_, v_ in METALS.items()}
choice = max(share, key=share.get)
assert choice == "caesium" and round(100 * share["caesium"]) == 79 and share["zinc"] == 0.0

# the sensor's numbers: blue light of 450 nm on caesium, 1 mW, one photon in a thousand freeing an electron
K_s = k_max(photon_ev(450.0), METALS["caesium"])                              # eV
I_s = photon_rate(1.0e-3, 450.0) / 1000 * E                                   # amps: electrons each second × e
print(f"The sensor: 1 mW of blue light (450 nm) on caesium is {photon_rate(1.0e-3, 450.0):.1e} photons a second. "
      f"K max = {K_s:.2f} eV; one photon in a thousand frees an electron: {I_s * 1e6:.2f} µA.")
assert round(K_s, 2) == 0.81 and round(I_s * 1e6, 2) == 0.36
assert agrees("P3.8", [k_max(photon_ev(450), 1.95), photon_rate(2.0e-3, 450) / 500 * E * 1e6])
assert agrees("P1.7", [threshold_nm(METALS["sodium"])]) and agrees("P2.10", [threshold_nm(METALS["potassium"])])

In [ ]:
interact(sensor_picture, light_nm=FloatSlider(value=450.0, min=320.0, max=700.0, step=5.0, description="light (nm)"));

## Notes on sources
Every exercise and set-up here is our own. The constants are CODATA 2022 values, rounded (h = 6.63 × 10⁻³⁴ J s =
4.14 × 10⁻¹⁵ eV s; c = 3.00 × 10⁸ m/s; e = 1.60 × 10⁻¹⁹ C; the electron's mass 9.11 × 10⁻³¹ kg; the proton's
1.67 × 10⁻²⁷ kg; hc = 1240 eV nm, from 1239.84). The three routes to a photon's energy differ in the third figure, so
every question names the constant to use. Work functions are typical values, and they vary with the surface: caesium
1.95, potassium 2.29, sodium 2.75 and zinc 3.63 eV, the lowest of zinc's listed 3.63 to 4.9 eV (CRC Handbook of
Chemistry and Physics, as tabulated in Wikipedia's "Work function", retrieved 2026-11-16). The photo-tube is a model:
flat plates, and a simple rule for the energy the electrons lose on the way out. It is there to show what Einstein's
equation predicts, not to replace a measurement, and the readings on the stopping-potential graph are illustrative.
The build-up of a two-slit pattern from single electrons was first shown by Merli, Missiroli and Pozzi (1976) and by
Tonomura and others (1989). The colours of the spectrum bar are approximate.

In [ ]:
# When this notebook runs beside the course's source files, this cell checks it still matches them; elsewhere it does
# nothing.
import importlib.util, pathlib, sys
_src = pathlib.Path("../../../episodes/ph21_dual_nature/practice.py")
if _src.exists():
    sys.path.insert(0, str(_src.resolve().parents[2]))
    _spec = importlib.util.spec_from_file_location("ph21_practice", _src)
    _P = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(_P)
    def _flat(a): return [y for x in a for y in _flat(x)] if isinstance(a, (tuple, list)) else [a]
    for _e in _P.EX:
        _q = Q[_e["id"]]
        assert (_q["q"], _q["working"], _q["function"], _q["tier"], _q["parts"], _q["hint"]) == (
            _e["q"], _e["working"], _e["function"], _e["tier"], _e["parts"], _e["hint"]), _e["id"]
        assert len(_q["answer"]) == len(_flat(_e["answer"])) and all(
            abs(a - float(b)) < 1e-9 * max(1, abs(b)) for a, b in zip(_q["answer"], _flat(_e["answer"]))), _e["id"]
    assert len(_P.EX) == len(EX)
    assert (H, H_EV, C, E, ME, MP, HC, EV) == (_P.H, _P.H_EV, _P.C, _P.E, _P.ME, _P.MP, _P.HC, _P.EV)
    assert METALS == _P.METALS and VISIBLE == _P.VISIBLE and SLIT == _P.SLIT
    assert (TUBE["seed"], TUBE["r_emit"], TUBE["r_coll"]) == (_P.TUBE["seed"], _P.TUBE["r_emit"], _P.TUBE["r_coll"])
    assert close(v0_k, _P.TEACH["v0"], 0.03) and close(v0_k, _P.SIM["stop_teach"], 0.03)
    assert close(band_6000, _P.SIM["band"], 1e-9) and close(strength_6000, _P.SIM["band_strength"], 1e-9)
    assert close(K_s, _P.SENSOR["k"], 1e-12) and close(I_s, _P.SENSOR["I"], 1e-12) and _P.HOOK_CHOICE == choice
    assert close(rate_g, _P.LASER["rate"], 1e-12) and close(lam_100, _P.AE["lam"], 1e-12) and close(ball, _P.BALL["lam"], 1e-12)
    print("matches the course's practice set")